# NLP Final Term — BERT Variation (RoBERTa)
Mirrors the group's BERT pipeline (same data prep, split, metrics, 8-run hyperparameter grid) but fine-tunes **RoBERTa (`roberta-base`)** instead of BERT.

**Crash-proof version:** every run's result is saved to Google Drive the moment it finishes. If Colab/Kaggle disconnects, just re-run the notebook from the top — completed runs are auto-skipped and only the remaining ones execute.


## 1. Load & Prepare Data


In [5]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split


In [6]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".json"):
            print(os.path.join(root, f))

/kaggle/input/datasets/tousiftasin/robert-grid-result/roberta_grid_results.json


In [7]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/tousiftasin/robert-grid-result/roberta_grid_results.json
/kaggle/input/datasets/tousiftasin/original-dataset/Original Dataset.csv


In [8]:
# --- Colab ---
# df = pd.read_csv("Original Dataset.csv")

# --- Kaggle (adjust path to match your uploaded dataset) ---
df = pd.read_csv("/kaggle/input/datasets/tousiftasin/original-dataset/Original Dataset.csv")

print(df.head())
print(df.shape)


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

In [9]:
print(df["category"].value_counts())


category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [10]:
selected_categories = [
    "POLITICS",
    "ENTERTAINMENT",
    "TRAVEL",
    "PARENTING"
]

df = df[df["category"].isin(selected_categories)]

print(df["category"].value_counts())
print(df.shape)


category
POLITICS         35602
ENTERTAINMENT    17362
TRAVEL            9900
PARENTING         8791
Name: count, dtype: int64
(71655, 6)


In [11]:
print("Missing Values:")
print(df.isnull().sum())


Missing Values:
link                    0
headline                1
category                0
short_description    6229
authors              9080
date                    0
dtype: int64


In [12]:
print("Duplicate Rows:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Shape after removing duplicates:", df.shape)


Duplicate Rows: 1
Shape after removing duplicates: (71654, 6)


In [13]:
df["text"] = (
    df["headline"].fillna("") + " " +
    df["short_description"].fillna("")
)

df = df[["text", "category"]]

print(df.head())
print(df.shape)


                                                 text       category
3   The Funniest Tweets From Parents This Week (Se...      PARENTING
20  Golden Globes Returning To NBC In January Afte...  ENTERTAINMENT
21  Biden Says U.S. Forces Would Defend Taiwan If ...       POLITICS
24  ‘Beautiful And Sad At The Same Time’: Ukrainia...       POLITICS
28  James Cameron Says He 'Clashed' With Studio Be...  ENTERTAINMENT
(71654, 2)


In [14]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text"] = df["text"].apply(clean_text)

print(df.head())


                                                 text       category
3   The Funniest Tweets From Parents This Week (Se...      PARENTING
20  Golden Globes Returning To NBC In January Afte...  ENTERTAINMENT
21  Biden Says U.S. Forces Would Defend Taiwan If ...       POLITICS
24  ‘Beautiful And Sad At The Same Time’: Ukrainia...       POLITICS
28  James Cameron Says He 'Clashed' With Studio Be...  ENTERTAINMENT


In [15]:
sample_size = 8791

df = (
    df.groupby("category", group_keys=False)
      .sample(n=sample_size, random_state=42)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(df["category"].value_counts())
print("Total samples:", len(df))


category
POLITICS         8791
TRAVEL           8791
ENTERTAINMENT    8791
PARENTING        8791
Name: count, dtype: int64
Total samples: 35164


In [16]:
print("Final Dataset Shape:", df.shape)
print("\nCategory Distribution:")
print(df["category"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


Final Dataset Shape: (35164, 2)

Category Distribution:
category
POLITICS         8791
TRAVEL           8791
ENTERTAINMENT    8791
PARENTING        8791
Name: count, dtype: int64

Missing Values:
text        0
category    0
dtype: int64

Duplicate Rows:
32


In [17]:
X = df["text"]
y = df["category"]

print("Input samples:", len(X))
print("Target samples:", len(y))


Input samples: 35164
Target samples: 35164


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 28131
Testing samples: 7033


In [19]:
print("Training Distribution:")
print(y_train.value_counts())

print("\nTesting Distribution:")
print(y_test.value_counts())


Training Distribution:
category
PARENTING        7033
TRAVEL           7033
ENTERTAINMENT    7033
POLITICS         7032
Name: count, dtype: int64

Testing Distribution:
category
POLITICS         1759
ENTERTAINMENT    1758
PARENTING        1758
TRAVEL           1758
Name: count, dtype: int64


## 2. Crash-Proof Result Tracking (Google Drive)
Mounts Drive and loads any results already completed in a previous session, so a disconnect never costs you finished runs.

**On Kaggle**, Drive mounting isn't available — this cell auto-detects that and falls back to saving inside `/kaggle/working/` instead (download that file periodically as backup, since Kaggle sessions aren't as persistent as Drive).


In [20]:
import os
import json

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_PATH = "/content/drive/MyDrive/roberta_grid_results.json"
else:
    RESULTS_PATH = "/kaggle/working/roberta_grid_results.json"

import shutil

uploaded_json_path = "/kaggle/input/datasets/tousiftasin/robert-grid-result/roberta_grid_results.json"  # <- paste your real path here

if os.path.exists(uploaded_json_path):
    shutil.copy(uploaded_json_path, RESULTS_PATH)
    print("Copied previous results into place.")
    
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        completed_results = json.load(f)
else:
    completed_results = {}

print("Results file:", RESULTS_PATH)
print("Already completed runs:", list(completed_results.keys()))


Copied previous results into place.
Results file: /kaggle/working/roberta_grid_results.json
Already completed runs: ['lr2e-05_bs16_wd0.01', 'lr3e-05_bs16_wd0.01', 'lr2e-05_bs32_wd0.01', 'lr3e-05_bs32_wd0.01', 'lr2e-05_bs16_wd0.1', 'lr3e-05_bs16_wd0.1']


## 3. RoBERTa Setup


In [21]:
!pip install -q transformers datasets accelerate


In [22]:
import numpy as np
import torch

from torch.utils.data import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [23]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)


Classes: ['ENTERTAINMENT' 'PARENTING' 'POLITICS' 'TRAVEL']


In [24]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Speed tip:** encodings below are *not* padded to a fixed length here. Instead we truncate to 128 tokens max and pad dynamically per-batch with `DataCollatorWithPadding` (next cell). Most headlines are much shorter than 128 tokens, so dynamic padding avoids wasting compute on padding tokens — this alone can cut training time noticeably without changing results.


In [25]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    max_length=128
)

print("Training texts:", len(train_encodings["input_ids"]))
print("Testing texts:", len(test_encodings["input_ids"]))


Training texts: 28131
Testing texts: 7033


In [26]:
class RoBERTaDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }

        item["labels"] = int(self.labels[index])

        return item


In [27]:
train_dataset = RoBERTaDataset(
    train_encodings,
    y_train_encoded
)

test_dataset = RoBERTaDataset(
    test_encodings,
    y_test_encoded
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))


Training samples: 28131
Testing samples: 7033


In [28]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


## 4. Hyperparameter Grid (Same 8 Combinations as the BERT Runs)
LR ∈ {2e-5, 3e-5}, Batch Size ∈ {16, 32}, Weight Decay ∈ {0.01, 0.1} — 5 epochs w/ early stopping, warmup ratio 0.1, max sequence length 128, dropout 0.1, AdamW.

Speed settings added: `fp16` (auto-enabled on GPU), dynamic padding via the data collator above, `dataloader_num_workers=2` for faster data loading, and `save_total_limit=1` so disk writes stay small.


In [29]:
def run_roberta_experiment(learning_rate, batch_size, weight_decay):

    print("\nStarting RoBERTa experiment...")
    print("Learning Rate:", learning_rate)
    print("Batch Size:", batch_size)
    print("Weight Decay:", weight_decay)

    model = RobertaForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=4,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1
    )

    num_train_samples = len(train_dataset)
    steps_per_epoch = int(np.ceil(num_train_samples / batch_size))
    total_steps = steps_per_epoch * 5
    warmup_steps = int(total_steps * 0.1)

    print("Warmup Steps:", warmup_steps)

    use_fp16 = torch.cuda.is_available()

    training_args = TrainingArguments(
        output_dir="./roberta_results",
        num_train_epochs=5,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="epoch",
        save_total_limit=1,
        fp16=use_fp16,
        dataloader_num_workers=2,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    results = trainer.evaluate()

    print("\nFinal Results:")
    print("Accuracy:", results["eval_accuracy"])
    print("Precision:", results["eval_precision"])
    print("Recall:", results["eval_recall"])
    print("F1 Score:", results["eval_f1"])

    return results


## 5. Run the Grid (Auto-Resumes on Reconnect)
Each config is skipped if it's already saved to `RESULTS_PATH` — so if this cell gets interrupted partway through, just re-run it and it'll pick up where it left off.


In [30]:
def run_config(lr, bs, wd):

    key = f"lr{lr}_bs{bs}_wd{wd}"

    if key in completed_results:
        print(f"Skipping {key} — already completed")
        return completed_results[key]

    result = run_roberta_experiment(
        learning_rate=lr,
        batch_size=bs,
        weight_decay=wd
    )

    completed_results[key] = {
        "learning_rate": lr,
        "batch_size": bs,
        "weight_decay": wd,
        "accuracy": result["eval_accuracy"],
        "precision": result["eval_precision"],
        "recall": result["eval_recall"],
        "f1": result["eval_f1"]
    }

    with open(RESULTS_PATH, "w") as f:
        json.dump(completed_results, f, indent=2)

    print(f"Saved {key} to {RESULTS_PATH}")

    return result


configs = [
    (0.00002, 16, 0.01),
    (0.00003, 16, 0.01),
    (0.00002, 32, 0.01),
    (0.00003, 32, 0.01),
    (0.00002, 16, 0.1),
    (0.00003, 16, 0.1),
    (0.00002, 32, 0.1),
    (0.00003, 32, 0.1),
]

for lr, bs, wd in configs:
    run_config(lr, bs, wd)


Skipping lr2e-05_bs16_wd0.01 — already completed
Skipping lr3e-05_bs16_wd0.01 — already completed
Skipping lr2e-05_bs32_wd0.01 — already completed
Skipping lr3e-05_bs32_wd0.01 — already completed
Skipping lr2e-05_bs16_wd0.1 — already completed
Skipping lr3e-05_bs16_wd0.1 — already completed

Starting RoBERTa experiment...
Learning Rate: 2e-05
Batch Size: 32
Weight Decay: 0.1


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Results:
Accuracy: 0.9618939286222096
Precision: 0.9619325064551439
Recall: 0.9618939286222096
F1 Score: 0.9618994206349268
Saved lr3e-05_bs32_wd0.1 to /kaggle/working/roberta_grid_results.json


## 6. Summary Table (Fill in the "Your BERT Variation" rows)


In [31]:
summary_rows = []

for key, res in completed_results.items():
    summary_rows.append({
        "Learning Rate": res["learning_rate"],
        "Batch Size": res["batch_size"],
        "Weight Decay": res["weight_decay"],
        "Accuracy": res["accuracy"],
        "Precision": res["precision"],
        "Recall": res["recall"],
        "F1": res["f1"]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["Learning Rate", "Batch Size", "Weight Decay"]).reset_index(drop=True)

print(summary_df.to_string(index=False))
summary_df.to_csv("roberta_grid_results.csv", index=False)


 Learning Rate  Batch Size  Weight Decay  Accuracy  Precision   Recall       F1
       0.00002          16          0.01  0.958766   0.958840 0.958766 0.958789
       0.00002          16          0.10  0.959335   0.959313 0.959335 0.959319
       0.00002          32          0.01  0.960756   0.960800 0.960756 0.960773
       0.00002          32          0.10  0.961610   0.961664 0.961610 0.961627
       0.00003          16          0.01  0.959335   0.959382 0.959335 0.959346
       0.00003          16          0.10  0.957628   0.957694 0.957628 0.957649
       0.00003          32          0.01  0.959335   0.959327 0.959335 0.959325
       0.00003          32          0.10  0.961894   0.961933 0.961894 0.961899


**Note:** the row with the highest F1 in `summary_df` is the one to report as "Your BERT Variation" in the final combined model-comparison table (alongside Naive Bayes, Logistic Regression, SVM, BERT, and the Ensemble).


## 7. Export Best RoBERTa Model Predictions (for Ensemble)
Finds the best config from summary_df, retrains it once more, and saves predicted probabilities on the test set to combine with BERT for the ensemble.


In [32]:
best_row = summary_df.sort_values("F1", ascending=False).iloc[0]

best_lr = best_row["Learning Rate"]
best_bs = int(best_row["Batch Size"])
best_wd = best_row["Weight Decay"]

print("Best RoBERTa config:")
print("Learning Rate:", best_lr)
print("Batch Size:", best_bs)
print("Weight Decay:", best_wd)
print("F1:", best_row["F1"])


Best RoBERTa config:
Learning Rate: 3e-05
Batch Size: 32
Weight Decay: 0.1
F1: 0.9618994206349268


In [33]:
best_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=4,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

num_train_samples = len(train_dataset)
steps_per_epoch = int(np.ceil(num_train_samples / best_bs))
total_steps = steps_per_epoch * 5
warmup_steps = int(total_steps * 0.1)

training_args = TrainingArguments(
    output_dir="./roberta_best_results",
    num_train_epochs=5,
    learning_rate=best_lr,
    per_device_train_batch_size=best_bs,
    per_device_eval_batch_size=best_bs * 2,
    weight_decay=best_wd,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=best_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

final_results = trainer.evaluate()
print("Confirm best RoBERTa results:", final_results)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along d

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.916560,0.396570,0.940850,0.943007,0.940850,0.941030
2,0.292263,0.288275,0.956917,0.957139,0.956917,0.956903
3,0.169766,0.291852,0.960472,0.960450,0.960472,0.960451
4,0.095790,0.369814,0.960756,0.960824,0.960756,0.960773
5,0.050050,0.410644,0.961894,0.961933,0.961894,0.961899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Confirm best RoBERTa results: {'eval_loss': 0.4106438457965851, 'eval_accuracy': 0.9618939286222096, 'eval_precision': 0.9619325064551439, 'eval_recall': 0.9618939286222096, 'eval_f1': 0.9618994206349268, 'eval_runtime': 24.9075, 'eval_samples_per_second': 282.365, 'eval_steps_per_second': 2.208, 'epoch': 5.0}


In [34]:
predictions_output = trainer.predict(test_dataset)

roberta_probs = torch.nn.functional.softmax(
    torch.tensor(predictions_output.predictions),
    dim=-1
).numpy()

np.save("roberta_test_probs.npy", roberta_probs)
np.save("roberta_test_labels.npy", y_test_encoded)

print("Saved roberta_test_probs.npy with shape:", roberta_probs.shape)
print("Saved roberta_test_labels.npy with shape:", y_test_encoded.shape)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Saved roberta_test_probs.npy with shape: (7033, 4)
Saved roberta_test_labels.npy with shape: (7033,)


In [35]:
# Colab
from google.colab import files
files.download("roberta_test_probs.npy")
files.download("roberta_test_labels.npy")

# Kaggle: instead, find both files in the Output panel on the right and click download


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>